In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


기존 모델 및 토크나이저 : model, tokenizer

학습 모델 및 토크나이저 : tuned_model, tuned_tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

os.environ["TRANSFORMERS_CACHE"] = "/content/drive/MyDrive/huggingface_cache"
# https://huggingface.co/nuprl/MultiPL-T-StarCoderBase_1b

model_name = "nuprl/MultiPLCoder-1b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
lua_revision = "7e96d931547e342ad0661cdd91236fe4ccf52545"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    revision=lua_revision,
    torch_dtype="auto",
    device_map="auto"
).cuda()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/532 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tuned_model_name = "bangill/maplestoryworlds-lua-api-finetune"
tuned_tokenizer = AutoTokenizer.from_pretrained(tuned_model_name)
tuned_model = AutoModelForCausalLM.from_pretrained(
    tuned_model_name,
    torch_dtype="auto",
    device_map="auto"
).cuda()

#### Lua 함수 자동완성 함수

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList
import torch

class StopOnSequences(StoppingCriteria):
    def __init__(self, stop_sequences, tokenizer):
        self.stop_ids = [tokenizer.encode(s, add_special_tokens=False) for s in stop_sequences]

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        seq = input_ids[0].tolist()
        for ids in self.stop_ids:
            if len(seq) >= len(ids) and seq[-len(ids):] == ids:
                return True
        return False

stop_words = ["\n\n", "print", "--", "function", "return", "end"]


def autocomplete_lua_function(model, tokenizer, prompt: str) -> str:
    stops = StoppingCriteriaList([StopOnSequences(stop_words, tokenizer)])

    # 모델에 입력
    output = model.generate(
        tokenizer(prompt, return_tensors="pt").input_ids.cuda(),
        max_new_tokens=20,
        do_sample=False,
        temperature=0.4,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
        stopping_criteria=stops,
        repetition_penalty=1.2,
    )

    # 토큰 → 텍스트
    generated_code = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"generated_code : \n{generated_code}\n")

    # 프롬프트 이후 부분만 추출
    generated_code = generated_code[len(prompt):] # 프롬프트 길이 만큼은 사용하지 않는다.
    # print(f"generated_code[len(prompt):] : \n{generated_code}\n")

    # 첫 번째 함수 단위만 가져오기 (Lua: function ... end)
    if "function" in generated_code:
        functions = generated_code.split("function")
        if len(functions) > 1:
            generated_code = "function" + functions[1].split("end")[0] + "end"
    # print(f"generated_code before return : \n{generated_code}\n")
    return generated_code.strip()


prompt : "# 두 수를 더하는 함수를 작성하시오. \nfunction add(a,b)\n return a + "

In [ ]:
# 기존 모델
print(autocomplete_lua_function(model,tokenizer, "# 두 수를 더하는 함수를 작성하시오. \nfunction add(a,b)\n return a + "))


generated_code : 
# 두 수를 더하는 함수를 작성하시오. 
function add(a,b)
 return a +  b end

b end


In [ ]:
# 학습 모델
print(autocomplete_lua_function(tuned_model,tuned_tokenizer, "# 두 수를 더하는 함수를 작성하시오. \nfunction add(a,b)\n return a + "))


generated_code : 
# 두 수를 더하는 함수를 작성하시오. 
function add(a,b)
 return a +  b
end

b
end


prompt : "local currentTargetEntity = self.Entity.AI"

In [ ]:
# 기존 모델
print(autocomplete_lua_function(model,tokenizer,"local currentTargetEntity = self.Entity.AI"))

generated_code : 
local currentTargetEntity = self.Entity.AI_TARGET
    if not target then
        return nil, "No Target"
    end
    
    local distance

_TARGET
    if not target then
        return nil, "No Target"
    end
    
    local distance


In [ ]:
# 학습 모델
print(autocomplete_lua_function(tuned_model,tuned_tokenizer, "local currentTargetEntity = self.Entity.AI"))

generated_code : 
local currentTargetEntity = self.Entity.AIChaseComponent:GetCurrentTarget()
if currentTargetEntity == nil thenreturn

ChaseComponent:GetCurrentTarget()
if currentTargetEntity == nil thenreturn


prompt : "local pages = _BadgeService:"

In [ ]:
# 기존 모델
print(autocomplete_lua_function(model,tokenizer,"local pages = _BadgeService:"))

generated_code : 
local pages = _BadgeService:get_badge_pages(user)
    if not pages then
        return nil, "No badge

get_badge_pages(user)
    if not pages then
        return nil, "No badge


In [ ]:
# 학습 모델
print(autocomplete_lua_function(tuned_model,tuned_tokenizer,"local pages = _BadgeService:"))

generated_code : 
local pages = _BadgeService:GetBadgesByUserId(userId)for i, badge in pairs(pages) do
	

GetBadgesByUserId(userId)for i, badge in pairs(pages) do


prompt : "self.ParticleComponent ="

In [ ]:
# 기존 모델
print(autocomplete_lua_function(model,tokenizer,"self.ParticleComponent ="))

generated_code : 
self.ParticleComponent = ParticleComponent
	end

ParticleComponent
	end


In [ ]:
# 학습 모델
print(autocomplete_lua_function(tuned_model,tuned_tokenizer,"self.ParticleComponent ="))

generated_code : 
self.ParticleComponent = self.Entity.ParticleComponent
if not self.ParticleComponent thenreturn

self.Entity.ParticleComponent
if not self.ParticleComponent thenreturn


prompt : "local playerController = self.Entity."

In [ ]:
# 기존 모델
print(autocomplete_lua_function(model,tokenizer,"local playerController = self.Entity."))

generated_code : 
local playerController = self.Entity.GetPlayer()
    if not player then
        return false, "No Player"
    end
    
    local

GetPlayer()
    if not player then
        return false, "No Player"
    end
    
    local


In [ ]:
# 학습 모델
print(autocomplete_lua_function(tuned_model,tuned_tokenizer,"local playerController = self.Entity."))

generated_code : 
local playerController = self.Entity.PlayerControllerComponent
if not playerController thenreturn

PlayerControllerComponent
if not playerController thenreturn
